# **Motivation**

# **Set Up**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

sns.set_style("whitegrid")

#Load data
os.chdir("/Users/leonorafonso/Documents/Work/ML/protein_ligand_binding_affinity_pipeline")
df_pchembl = pd.read_csv("data/processed/features_processed_pchembl.csv") 
df_pki = pd.read_csv("data/processed/features_processed_pki.csv")

# **Duplicate Analysis**

## **pChEMBL**

In [ ]:
dup_cols = ['UniProt_ID', 'CANONICAL_SMILES']

duplicates = df_pchembl.duplicated(subset=dup_cols)
print(f"Exact duplicates pChEMBL: {duplicates.sum()}")

df_pchembl[duplicates].head()

## **pKi**

In [ ]:
dup_cols = ['UniProt_ID', 'CANONICAL_SMILES']

duplicates = df_pki.duplicated(subset=dup_cols)
print(f"Exact duplicates pKi: {duplicates.sum()}")

df_pki[duplicates].head()

# **Ligand Leakage**

## **pChEMBL**

In [ ]:
# Same molecule different rows
ligand_counts = df_pchembl['CANONICAL_SMILES'].value_counts()

plt.figure()
ligand_counts.head(20).plot(kind='bar')
plt.title("Top repeated ligands pChEMBL")
plt.show()

# Chemical Similarity Leakage
chem_features = ['MolWt', 'MolLogP', 'TPSA']

sample = df_pchembl[chem_features].dropna().sample(1000, random_state=42)

sim_matrix = cosine_similarity(sample)

# Remove diagonal
np.fill_diagonal(sim_matrix, 0)

high_sim = (sim_matrix > 0.95).sum()
print(f"Highly similar ligand pairs pChEMBL (>0.95): {high_sim}")

## **pKi**

In [ ]:
# Same molecule different rows
ligand_counts = df_pki['CANONICAL_SMILES'].value_counts()

plt.figure()
ligand_counts.head(20).plot(kind='bar')
plt.title("Top repeated ligands pKi")
plt.show()

# Chemical Similarity Leakage
chem_features = ['MolWt', 'MolLogP', 'TPSA']

sample = df_pki[chem_features].dropna().sample(1000, random_state=42)

sim_matrix = cosine_similarity(sample)

# Remove diagonal
np.fill_diagonal(sim_matrix, 0)

high_sim = (sim_matrix > 0.95).sum()
print(f"Highly similar ligand pairs pKi (>0.95): {high_sim}")

# **Protein Leakage** 

## **pChEMBL**

In [ ]:
protein_counts = df_pchembl['UniProt_ID'].value_counts()

plt.figure()
protein_counts.head(20).plot(kind='bar')
plt.title("Most frequent proteins pChEMBL")
plt.show()

print("Unique proteins:", df_pchembl['UniProt_ID'].nunique())

# Embedding Similarity Leakage
esm_cols = [c for c in df_pchembl.columns if c.startswith('ESM2_')]

sample_esm = df_pchembl.dropna(subset=esm_cols).sample(1000, random_state=42)

emb = sample_esm[esm_cols].values

sim = cosine_similarity(emb)
np.fill_diagonal(sim, 0)

print("Highly similar proteins pChEMBL (>0.98):", (sim > 0.98).sum())

## **pKi**

In [ ]:
protein_counts = df_pKi['UniProt_ID'].value_counts()

plt.figure()
protein_counts.head(20).plot(kind='bar')
plt.title("Most frequent proteins pKi")
plt.show()

print("Unique proteins:", df_pki['UniProt_ID'].nunique())

# Embedding Similarity Leakage
esm_cols = [c for c in df_pki.columns if c.startswith('ESM2_')]

sample_esm = df_pki.dropna(subset=esm_cols).sample(1000, random_state=42)

emb = sample_esm[esm_cols].values

sim = cosine_similarity(emb)
np.fill_diagonal(sim, 0)

print("Highly similar proteins pKi (>0.98):", (sim > 0.98).sum())

# **Assay Leakage**

## **pChEMBL**

In [ ]:
assay_counts = df_pchembl['target_chembl_id'].value_counts()

plt.figure()
assay_counts.head(20).plot(kind='bar')
plt.title("Most frequent targets pChEMBL (ChEMBL)")
plt.show()

# Check variance within same assay
grouped = df.groupby('target_chembl_id')['pchembl_value'].agg(['mean', 'std', 'count'])

display(grouped.sort_values('count', ascending=False).head())

## **pKi**

In [ ]:
assay_counts = df_pki['target_chembl_id'].value_counts()

plt.figure()
assay_counts.head(20).plot(kind='bar')
plt.title("Most frequent targets pKi (ChEMBL)")
plt.show()

# Check variance within same assay
grouped = df_pki.groupby('target_chembl_id')['pKi'].agg(['mean', 'std', 'count'])

display(grouped.sort_values('count', ascending=False).head())

# **Cross Leakage**

## **pChEMBL**

In [ ]:
pair_counts = df_pchembl.groupby(['UniProt_ID', 'CANONICAL_SMILES']).size()

repeated_pairs = pair_counts[pair_counts > 1]

print(f"Repeated ligand-protein pairs pChEMBL: {len(repeated_pairs)}")

## **pKi**

In [ ]:
pair_counts = df_pki.groupby(['UniProt_ID', 'CANONICAL_SMILES']).size()

repeated_pairs = pair_counts[pair_counts > 1]

print(f"Repeated ligand-protein pairs pKi: {len(repeated_pairs)}")

# **Simulated Splits**

## **pChEMBL**

## **pKi**